# DistilBERT Classifier with Bayesian Optimisation

### Cleaning Dataset
To ensure consistency across the different models, our group will be using the same method of preprocessing and data cleaning methodology.  
Data cleaning was done with the following:
- Expanded contractions
- Removed Hashtags
- Removed airline mentions and kept their names
- Removed links
- Converted Emojis to texts


Importing Libaries and Downloading Modules

In [4]:
%pip install contractions
%pip install emoji
%pip install nltk

import pandas as pd
import contractions  # Import contractions package for contraction expansion
import re
import emoji
import nltk
from nltk.stem import WordNetLemmatizer  # Word lemmatization

nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Load the Dataset

In [5]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/tanhonjung/tweets-csv/Tweets.csv")
df


,airline_sentiment,sentiment_confidence,text
0,neutral,1.0000,@VirginAmerica What @dhepburn said.
1,positive,0.3486,@VirginAmerica plus you've added commercials t...
2,neutral,0.6837,@VirginAmerica I didn't today... Must mean I n...
3,negative,1.0000,@VirginAmerica it's really aggressive to blast...
4,negative,1.0000,@VirginAmerica and it's a really big bad thing...
...,...,...,...
14634,positive,0.3487,@AmericanAir thank you we got on a different f...
14635,negative,1.0000,@AmericanAir leaving over 20 minutes Late Flig...
14636,neutral,1.0000,@AmericanAir Please bring American Airlines to...
14637,negative,1.0000,"@AmericanAir you have my money, you change my ..."


In [6]:
# Define the list of airlines (case-insensitive)
airline_list = ["VirginAmerica", "united", "SouthwestAir", "JetBlue", "USAirways", "AmericanAir"]

def clean_text(text):

    # Convert emojis to text descriptions
    text = emoji.demojize(text, delimiters=(" ", " "))  # 😊 → " smiley face "

    # Expand contractions (e.g., "can't" → "cannot")
    text = contractions.fix(text)

    # Remove hashtags but keep words (e.g., "#happy" → "happy")
    text = re.sub(r'#(\w+)', r'\1', text)

    # Extract airline name if mentioned
    airline_pattern = r'@(' + '|'.join(airline_list) + r')\b'
    match = re.search(airline_pattern, text, re.IGNORECASE)
    airline = match.group(1) if match else "Unknown"

    # Remove airline mentions
    text = re.sub(airline_pattern, '', text, flags=re.IGNORECASE).strip()

    # Detect and remove hyperlinks
    link_pattern = r'http\S+'
    has_link = 1 if re.search(link_pattern, text) else 0
    text = re.sub(link_pattern, '', text).strip()

    # Remove punctuation and convert to lowercase
    text = re.sub(r'[^\w\s]', '', text).lower()

    # Remove underscores
    text = text.replace('_', ' ')
    text = text.replace('-', ' ')

    # Apply lemmatization
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words]
    text = " ".join(words)

    return airline, has_link, text

# Apply function and create new columns
df[['airline', 'has_link', 'clean_text']] = df['text'].apply(lambda x: pd.Series(clean_text(x)))


# Display first few rows of the emoji-converted text
df[380:391]


,airline_sentiment,sentiment_confidence,text,airline,has_link,clean_text
380,positive,1.0000,@VirginAmerica gave a credit for my Late Fligh...,VirginAmerica,0,gave a credit for my late flight flight yester...
381,neutral,1.0000,@VirginAmerica I need a receipt for a flight c...,VirginAmerica,0,i need a receipt for a flight change can you s...
382,negative,1.0000,"@VirginAmerica, I submitted a status match req...",VirginAmerica,0,i submitted a status match request a while bac...
383,positive,1.0000,@VirginAmerica had me at their safety video . ...,VirginAmerica,1,had me at their safety video loved my first cr...
384,positive,0.6871,@VirginAmerica that doesn't look to fat to me!...,VirginAmerica,0,that doe not look to fat to me it look yummy
385,neutral,1.0000,@VirginAmerica CEO says #Southwest &amp; #jetb...,VirginAmerica,1,ceo say southwest amp jetblue have strayed fro...
386,neutral,0.6811,@VirginAmerica a brilliant brisk am in Boston ...,VirginAmerica,1,a brilliant brisk am in boston in cue for vx363
387,neutral,1.0000,@VirginAmerica Atlantic ploughs a lone furrow ...,VirginAmerica,1,atlantic plough a lone furrow in the middleeas...
388,neutral,1.0000,@VirginAmerica Atlantic ploughs a lone furrow ...,VirginAmerica,1,atlantic plough a lone furrow in the middleeas...
389,neutral,0.7026,@VirginAmerica Atlantic ploughs a lone furrow ...,VirginAmerica,1,atlantic plough a lone furrow in the middleeas...


## Sentiment Classification using DistilBert Classifier


For this Deep Learning method, we wil be using transfer learning using a pre-trained model, specifically DistilBert with fine-tuning. DistilBert is a faster, cheaper and lighter transformer model based on the Bert architecture (Sanh et al., 2019), making it less resource intensive while retaining most of its predictive capabilities.


This approach draws inspiration from (Akpatsa et al., 2022) whereby researchers used both Bert and DistilBert models to perform Online News Sentiment Classification. The experiments confirmed the superiority of the transformer-based (BERT, and DistilBERT) models over other Machine Learning on a downstream NLP task, which is what we will try to reproduce in our study.



---



Before being passed into the code, sentiment classifications need to be encoded into numerical values such as {0: negative, 1: neutral, 2: positive}, a format which the algorithm understands.

We then use DistilBertTokenizer to split raw text data into smaller units called tokens, making it suitable for processing by the DistilBert model.

We have also used a train-test split of 80:20 for model validation.

In [7]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer

import torch
import numpy as np

# Encode labels (sentiment categories)
label_encoder = LabelEncoder()
df['encoded_sentiment'] = label_encoder.fit_transform(df['airline_sentiment'])

# Split into features (X) and target (y)
X = df['clean_text']
y = df['encoded_sentiment']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize and pad the text data
X_train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
X_test_encodings = tokenizer(X_test.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')

# Convert the labels to tensors
y_train_tensor = torch.tensor(y_train.values)
y_test_tensor = torch.tensor(y_test.values)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The study also show that a class imbalance in the dataset can affect model performance (Akpatsa et al., 2022). Keeping this in mind, we will use RandomOverSampler to address imbalanced datasets by randomly duplicating examples from the minority class to balance the class distribution.

In [8]:
from imblearn.over_sampling import RandomOverSampler

input_ids_np = X_train_encodings['input_ids'].numpy()
attention_mask_np = X_train_encodings['attention_mask'].numpy()
labels_np = y_train_tensor.numpy()

# Combine input_ids and attention_mask
X_combined = np.concatenate([input_ids_np, attention_mask_np], axis=1)

# Apply RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_combined, labels_np)

# Split resampled data
seq_len = X_train_encodings['input_ids'].shape[1]
input_ids_resampled = X_resampled[:, :seq_len]
attention_mask_resampled = X_resampled[:, seq_len:]

# Convert back to tensors
input_ids_tensor = torch.tensor(input_ids_resampled, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask_resampled, dtype=torch.long)
y_resampled_tensor = torch.tensor(y_resampled, dtype=torch.long)

Define key model parameters and check oversampled training set size using ```len(train_dataset)```

Improving on the study, we used AdamW optimizer instead of Adam, which has been shown to yield better results in NLP and leads to better generalization and stability (Zhou et al., 2024).

In [9]:

from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score


# Define the DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

# Create TensorDataset for training
train_dataset = TensorDataset(input_ids_tensor, attention_mask_tensor, y_resampled_tensor)
test_dataset = TensorDataset(X_test_encodings['input_ids'], X_test_encodings['attention_mask'], y_test_tensor)

# Create DataLoader for training and validation
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# Set up the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Oversampled training set size:", len(train_dataset))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Oversampled training set size: 22014


Without Hyperparameter Tuning

In [10]:
# Training loop
def train_model(model, train_loader, optimizer, device, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}, Loss: {avg_loss}")

# Train the model
train_model(model, train_loader, optimizer, device)

# Save model
model.save_pretrained('/content/drive/MyDrive/base_model')

Epoch 1, Loss: 0.41541347187824634
Epoch 2, Loss: 0.15070154592442206
Epoch 3, Loss: 0.07983890532722039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation Function

In [11]:
from sklearn.metrics import classification_report, f1_score

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Accuracy
    f1 = f1_score(all_labels, all_preds, average = 'weighted')

    # Classification report
    report = classification_report(all_labels, all_preds, digits=4)

    return f1, report

# Usage
f1, report = evaluate_model(model, test_loader, device)

print(f"f1_score: {f1 * 100:.2f}%")
print("Classification Report:\n")
print(report)

f1_score: 82.04%
Classification Report:

              precision    recall  f1-score   support

           0     0.8820    0.9016    0.8917      1840
           1     0.6672    0.6703    0.6688       634
           2     0.7829    0.7070    0.7431       454

    accuracy                         0.8214      2928
   macro avg     0.7774    0.7597    0.7678      2928
weighted avg     0.8201    0.8214    0.8204      2928



One limitation of the study is that it uses a Stochastic Gradient Descent with Restart (SGDR) policy learning rate, fine-tuning only the learning rate parameter.

To improve on this, we used Bayesian Optimization as our hyperparameter tuning method, in order to find the best values for other parameters including batch size and number of epochs.



In [12]:
!pip install optuna
import optuna
from sklearn.metrics import f1_score

# Define the objective function for optimization
def objective(trial):
    # Suggest hyperparameters for optimization
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)  # Learning rate (log scale)
    batch_size = trial.suggest_int('batch_size', 8, 32)  # Batch size
    epochs = trial.suggest_int('epochs', 3, 5)  # Number of epochs

    # Load DistilBERT model
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

    # Set up optimizer
    optimizer = AdamW(model.parameters(), lr=lr)

    # Move model to device (GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Training loop
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}, Loss: {avg_loss}")

    # Evaluate the model using F1-score
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate F1-score
    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f"F1-Score: {f1 * 100:.2f}%")

    return f1  # Return the F1-score for optimization


study = optuna.create_study(direction="maximize")  # maximize F1-score
study.optimize(objective, n_trials=10)  # adjust n_trials for more/less iterations

# Get the best hyperparameters and evaluation score
print(f"Best hyperparameters: {study.best_params}")
print(f"Best F1-score: {study.best_value * 100:.2f}%")

model.save_pretrained('/content/drive/MyDrive/best_model')

[I 2026-03-05 05:48:10,000] A new study created in memory with name: no-name-8142c21b-da8f-4a4f-8d0a-3e2f82e2befc
/tmp/ipykernel_55/3154266644.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)  # Learning rate (log scale)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 0.4714847458039172
Epoch 2, Loss: 0.21319840179193159
Epoch 3, Loss: 0.1357615461547962
Epoch 4, Loss: 0.10641117676497616


[I 2026-03-05 05:58:32,581] Trial 0 finished with value: 0.7887272537009228 and parameters: {'lr': 0.00011975284002681021, 'batch_size': 9, 'epochs': 4}. Best is trial 0 with value: 0.7887272537009228.


...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 0.516744087455621
Epoch 2, Loss: 0.27818374041208
Epoch 3, Loss: 0.5375744398943213
Epoch 4, Loss: 0.3036147344987406
Epoch 5, Loss: 0.19168184372138902


[I 2026-03-05 07:34:09,546] Trial 9 finished with value: 0.7715239885709899 and parameters: {'lr': 0.00018993160353454215, 'batch_size': 17, 'epochs': 5}. Best is trial 4 with value: 0.8297036884397633.


F1-Score: 77.15%
Best hyperparameters: {'lr': 1.4943402932789462e-05, 'batch_size': 29, 'epochs': 3}
Best F1-score: 82.97%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best Model

In [13]:
# Retrieve the best hyperparameters
best_params = study.best_params

# Load the best model
best_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
best_optimizer = AdamW(best_model.parameters(), lr=best_params['lr'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)

# Create DataLoader for training with best batch size
best_train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)

# Retrain the model using best hyperparameters
train_model(best_model, best_train_loader, best_optimizer, device)

# Evaluate using evaluation function defined above
f1, report = evaluate_model(best_model, test_loader, device)

print(f"Final Evaluation using Best Hyperparameters:")
print(f"F1-Score: {f1 * 100:.2f}%")
print("Classification Report:\n")
print(report)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 0.4873110924308237
Epoch 2, Loss: 0.2144232505221704
Epoch 3, Loss: 0.10728748718239857
Final Evaluation using Best Hyperparameters:
F1-Score: 82.07%
Classification Report:

              precision    recall  f1-score   support

           0     0.9043    0.8679    0.8857      1840
           1     0.6245    0.7161    0.6672       634
           2     0.7885    0.7555    0.7717       454

    accuracy                         0.8176      2928
   macro avg     0.7724    0.7798    0.7749      2928
weighted avg     0.8258    0.8176    0.8207      2928



## References


---

Akpatsa, S.K., Lei, H., Li, X., Setornyo Obeng, V.K., Martey, E.M. et al. (2022). Online News Sentiment Classification Using DistilBERT. Journal of Quantum Computing, 4(1), 1–11. https://doi.org/10.32604/jqc.2022.026658

Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019, October 2). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. arXiv.org. https://arxiv.org/abs/1910.01108v4

Zhou, P., Xie, X., Lin, Z., & Yan, S. (2024). Towards understanding convergence and generalization of AdamW. IEEE Transactions on Pattern Analysis and Machine Intelligence. 1-8. https://ink.library.smu.edu.sg/sis_research/8986
